# OData Query Validator — Full Demo
Steps: empty check → syntax → safety → $top limits → expand depth → filter complexity → cost estimation

In [5]:
import sys, json
sys.path.insert(0, '..')

from validation import ODataQueryValidator, ValidatorConfig

validator = ODataQueryValidator(config=ValidatorConfig.for_production())

def show(label, result):
    status = ' VALID' if result.is_valid else ' INVALID'
    print(f'[{status}]  {label}')
    for issue in result.issues:
        icon = {'critical': '', 'warning': '', 'info': ''}.get(issue.severity.value, '')
        print(f'   {icon} [{issue.category}] {issue.message}')
        if issue.suggestion:
            print(f'       {issue.suggestion}')
    if result.estimated_cost_score is not None:
        print(f'   → cost score: {result.estimated_cost_score:.1f}/100')
    if result.estimated_row_count is not None:
        print(f'   → estimated rows: {result.estimated_row_count}')
    print()

print('Validator ready  (production config)')
print(f'  max $top             : {validator.config.max_retrieval_limit}')
print(f'  warn $top threshold  : {validator.config.warn_retrieval_threshold}')
print(f'  max expand depth     : {validator.config.max_expand_depth}')
print(f'  max filter complexity: {validator.config.max_filter_complexity}')
print(f'  max cost score       : {validator.config.max_cost_score}')
print(f'  warn cost threshold  : {validator.config.warn_cost_threshold}')

Validator ready  (production config)
  max $top             : 500
  warn $top threshold  : 250
  max expand depth     : 2
  max filter complexity: 50
  max cost score       : 80
  warn cost threshold  : 60


## Step 1 — Empty query check

In [4]:
show('Empty string',    validator.validate(''))
show('Whitespace only', validator.validate('     '))
show('None',            validator.validate(None))

[ INVALID]  Empty string
    [generation] No OData query was generated
      💡 Check query generation logic and prompts

[ INVALID]  Whitespace only
    [generation] No OData query was generated
      💡 Check query generation logic and prompts

[ INVALID]  None
    [generation] No OData query was generated
      💡 Check query generation logic and prompts



## Step 2 — Syntax check

In [12]:
show('Valid syntax',               validator.validate('Users?$top=10&$filter=Age gt 18'))
show('Unmatched parentheses',      validator.validate('Users?$top=10&$filter=(Age gt 18 and (Status eq \'Active\''))
show('Empty parameter value',      validator.validate('Users?$top=&$filter=Age gt 18'))
show('Unknown $option (warning)',  validator.validate('-----'))

# Note: the validator inspects OData $ parameters only.
# A free-text string like 'hello world' has no $ parameters so no syntax issues are raised —
# it is treated as an entity name with no query options.

[ VALID]  Valid syntax
   → cost score: 0.6/100
   → estimated rows: 10

[ INVALID]  Unmatched parentheses
    [syntax] Unmatched parentheses in $filter expression
       Check filter expression for balanced parentheses
   → cost score: 2.3/100
   → estimated rows: 10

[ INVALID]  Empty parameter value
    [syntax] Empty value for parameter: $top
       Provide a value for $top or remove the parameter
    [limits] Invalid $top value:  (must be an integer)
       Use a numeric value for $top
   → cost score: 0.2/100
   → estimated rows: 

[ INVALID]  Unknown $option (warning)
    [syntax] Query does not begin with a valid entity set name or OData parameter
       OData queries must start with an entity set name (e.g. Users?$top=10) or a $ parameter
    [limits] Query does not specify $top limit
       Add $top parameter (max: 500)
   → cost score: 40.0/100



## Step 3 — Safety check

In [13]:
show('Clean query',       validator.validate('Users?$top=10&$select=Name'))
show('Mutation keyword',  validator.validate('Users?$filter=Name eq \'test\' DELETE'))
show('SQL injection',     validator.validate('Users?$filter=Name eq \'x\' UNION SELECT * FROM Passwords'))
show('Script injection',  validator.validate('Users?$filter=Name eq \'<script>alert(1)</script>\''))
show('$batch blocked',    validator.validate('$batch?$top=10'))

[ VALID]  Clean query
   → cost score: 0.3/100
   → estimated rows: 10

[ INVALID]  Mutation keyword
    [safety] Detected unsafe operation keyword: DELETE
       Only SELECT/read operations are allowed
    [limits] Query does not specify $top limit
       Add $top parameter (max: 500)
   → cost score: 40.2/100

[ INVALID]  SQL injection
    [security] Detected potential SQL injection pattern
       Remove suspicious patterns from query
    [limits] Query does not specify $top limit
       Add $top parameter (max: 500)
   → cost score: 40.2/100

[ INVALID]  Script injection
    [security] Detected potential SQL injection pattern
       Remove suspicious patterns from query
    [security] Detected potential script injection: <script
       Remove script-like patterns from filter
    [limits] Query does not specify $top limit
       Add $top parameter (max: 500)
   → cost score: 40.8/100

[ INVALID]  $batch blocked
    [safety] $batch operations are disabled
       Execute queries indivi

## Step 4 — Retrieval limit check

In [14]:
show('$top=10   (fine)',    validator.validate('Users?$top=10'))
show('$top=600  (warning)', validator.validate('Users?$top=600'))
show('$top=5000 (exceeds)', validator.validate('Users?$top=5000'))
show('$top=-1   (invalid)', validator.validate('Users?$top=-1'))
show('$top=abc  (invalid)', validator.validate('Users?$top=abc'))
show('missing $top',        validator.validate('Users?$filter=Age gt 18'))

[ VALID]  $top=10   (fine)
   → cost score: 0.3/100
   → estimated rows: 10

[ INVALID]  $top=600  (warning)
    [limits] $top value 600 exceeds maximum allowed: 500
       Reduce $top to 500 or less
   → cost score: 18.0/100
   → estimated rows: 600

[ INVALID]  $top=5000 (exceeds)
    [limits] $top value 5000 exceeds maximum allowed: 500
       Reduce $top to 500 or less
   → cost score: 30.0/100
   → estimated rows: 5000

[ INVALID]  $top=-1   (invalid)
    [limits] Invalid $top value: -1 (must be positive)
       Use a positive integer for $top
   → cost score: -0.0/100
   → estimated rows: -1

[ INVALID]  $top=abc  (invalid)
    [limits] Invalid $top value: abc (must be an integer)
       Use a numeric value for $top
   → cost score: 0.0/100

[ VALID]  missing $top
    [limits] Query does not specify $top limit
       Add $top parameter (max: 500)
   → cost score: 40.2/100



## Step 5 — Expand depth check

In [15]:
show('Depth 1 (fine)',    validator.validate('Orders?$top=10&$expand=OrderItems'))
show('Depth 2 (fine)',    validator.validate('Orders?$top=10&$expand=OrderItems($expand=Product)'))
show('Depth 4 (exceeds)', validator.validate('Orders?$top=10&$expand=OrderItems($expand=Product($expand=Category($expand=Supplier)))'))

[ VALID]  Depth 1 (fine)
   → cost score: 15.3/100
   → estimated rows: 10

[ VALID]  Depth 2 (fine)
   → cost score: 30.3/100
   → estimated rows: 10

[ INVALID]  Depth 4 (exceeds)
    [cost] $expand depth 4 exceeds maximum: 2
       Reduce nesting depth to 2 or less
    [cost] Query has moderate cost: 60.3/100 (limit: 80)
       Monitor performance and consider optimizations if slow
   → cost score: 60.3/100
   → estimated rows: 10



## Step 6 — Filter complexity check

In [16]:
show('Simple filter (fine)',
     validator.validate('Products?$top=50&$filter=Price gt 100'))

show('Moderate filter (fine)',
     validator.validate('Products?$top=50&$filter=Price gt 100 and Category eq \'Electronics\' and InStock eq true'))

# Complexity breakdown:
#   and x7 (+14), or x3 (+6), comparisons x11 (+11),
#   functions: contains+startswith+endswith+year+month (+15), parentheses x4 (+8) = 54 → exceeds limit of 50
show('Complex filter (exceeds — INVALID)',
     validator.validate(
         'Products?$top=50'
         '&$filter=(Price gt 100 and Price lt 500 and Price ne 200) or '
         '(contains(Name, \'laptop\') and startswith(Category, \'Electronics\') and endswith(Brand, \'Corp\')) or '
         '(Rating gt 4 and Rating le 5 and InStock eq true) or '
         '(year(OrderDate) eq 2024 and month(OrderDate) ge 1)'
     ))

[ VALID]  Simple filter (fine)
   → cost score: 1.8/100
   → estimated rows: 50

[ VALID]  Moderate filter (fine)
   → cost score: 3.2/100
   → estimated rows: 50

[ INVALID]  Complex filter (exceeds — INVALID)
    [cost] Filter complexity score 61 exceeds maximum allowed: 50
       Simplify the filter expression or break into multiple queries
   → cost score: 16.8/100
   → estimated rows: 50



## Step 7 — Cost estimation

In [20]:
# Low cost: $top=10 → 0.3 pts.  Total ≈ 0.3 (well below 60)
show('Low cost — small select (fine)',
     validator.validate('Users?$top=10&$select=Name,Email'))

# Medium cost: no $top (+40) + 1 expand level (+15) + orderby (+10) = 65  → WARNING (60–80)
show('Medium cost — warning, not blocked',
     validator.validate('Products?$expand=Category&$orderby=Name'))

# High cost: no $top (+40) + 2 expand levels (+30) + orderby (+10) + count (+5) = 85 → CRITICAL (> 80)
show('High cost — exceeds limit, INVALID',
     validator.validate('Products?$expand=Category($expand=Supplier)&$orderby=Name&$count=true'))

[ VALID]  Low cost — small select (fine)
   → cost score: 0.3/100
   → estimated rows: 10

[ VALID]  Medium cost — warning, not blocked
    [limits] Query does not specify $top limit
       Add $top parameter (max: 500)
    [cost] Query has moderate cost: 65.0/100 (limit: 80)
       Monitor performance and consider optimizations if slow
   → cost score: 65.0/100

[ INVALID]  High cost — exceeds limit, INVALID
    [cost] Query cost score 85.0/100 exceeds maximum allowed (80)
       Consider adding filters, reducing $top, or limiting $expand depth
    [limits] Query does not specify $top limit
       Add $top parameter (max: 500)
   → cost score: 85.0/100



## All checks run simultaneously — multi-failure examples

Every query passes through all 7 steps regardless. If multiple rules are broken, all failures are reported together.

In [18]:
# Fails Step 3 (SQL injection) + Step 4 ($top is negative) — two independent critical issues
show('Fail: safety + limits',
     validator.validate(
         "Users?$top=-1&$filter=Name eq 'x' UNION SELECT * FROM Passwords"
     ))

# Fails Step 4 ($top too high) + Step 5 (expand too deep) + Step 7 (cost warning)
show('Fail: $top limit + expand depth + cost warning',
     validator.validate(
         'Orders?$top=5000&$expand=OrderItems($expand=Product($expand=Category))'
     ))

[ INVALID]  Fail: safety + limits
    [security] Detected potential SQL injection pattern
       Remove suspicious patterns from query
    [limits] Invalid $top value: -1 (must be positive)
       Use a positive integer for $top
   → cost score: 0.2/100
   → estimated rows: -1

[ INVALID]  Fail: $top limit + expand depth + cost warning
    [limits] $top value 5000 exceeds maximum allowed: 500
       Reduce $top to 500 or less
    [cost] $expand depth 3 exceeds maximum: 2
       Reduce nesting depth to 2 or less
    [cost] Query has moderate cost: 75.0/100 (limit: 80)
       Monitor performance and consider optimizations if slow
   → cost score: 75.0/100
   → estimated rows: 5000



## Passes steps 1–5, fails only at Step 6 (filter complexity)

A well-formed query — correct $top, no expand, no safety issues — that trips only the filter complexity limit.

In [21]:
# Step 1 — not empty           ✓
# Step 2 — syntax fine          ✓
# Step 3 — no safety issues     ✓
# Step 4 — $top=10 well within limit ✓
# Step 5 — no $expand           ✓
# Step 6 — complexity score 61 > 50  ✗  INVALID
# Step 7 — cost score ~16 (no issue, since step 6 already failed)
show('Passes steps 1-5, fails at Step 6 ',
     validator.validate(
         'Products?$top=10'
         '&$filter=(Price gt 100 and Price lt 500 and Price ne 200) or '
         '(contains(Name, \'laptop\') and startswith(Category, \'Electronics\') and endswith(Brand, \'Corp\')) or '
         '(Rating gt 4 and Rating le 5 and InStock eq true) or '
         '(year(OrderDate) eq 2024 and month(OrderDate) ge 1)'
     ))

[ INVALID]  Passes steps 1-5, fails at Step 6 
    [cost] Filter complexity score 61 exceeds maximum allowed: 50
       Simplify the filter expression or break into multiple queries
   → cost score: 15.6/100
   → estimated rows: 10

